# **Imports**

In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

import torchvision
import torchvision.datasets as datasets

import time

import cv2

import json
import numpy as np

import sys
import logging as log

In [2]:
from torchvision import models, datasets, transforms as T
from PIL import Image

## **Initialize env and drivers**

In [3]:
%cd /opt/intel/openvino_2022/

/opt/intel/openvino_2022.1.0.643


In [4]:
%ls install_dependencies/

install_NCS_udev_rules.sh*  install_openvino_dependencies.sh*
install_NEO_OCL_driver.sh*


In [5]:
!./setupvars.sh*

[setupvars.sh] OpenVINO environment initialized


In [6]:
!./install_dependencies/install_openvino_dependencies.sh

Detected OS: ubuntu20.04
Get:1 http://security.ubuntu.com/ubuntu focal-security InRelease [128 kB]
Hit:2 http://archive.ubuntu.com/ubuntu focal InRelease                         
Get:3 http://archive.ubuntu.com/ubuntu focal-updates InRelease [128 kB]        
Get:4 http://archive.ubuntu.com/ubuntu focal-backports InRelease [128 kB]
Fetched 383 kB in 1s (327 kB/s)    
Reading package lists... Done
Reading package lists... Done
Building dependency tree       
Reading state information... Done
g++ is already the newest version (4:9.3.0-1ubuntu2).
gcc is already the newest version (4:9.3.0-1ubuntu2).
libusb-1.0-0 is already the newest version (2:1.0.23-2build1).
make is already the newest version (4.2.1-1.2).
python3 is already the newest version (3.8.2-0ubuntu2).
python3-dev is already the newest version (3.8.2-0ubuntu2).
python3-venv is already the newest version (3.8.2-0ubuntu2).
cmake is already the newest version (3.16.3-1ubuntu1.20.04.1).
curl is already the newest version (7.68.0-1ub

In [7]:
!./install_dependencies/install_NCS_udev_rules.sh

Updating udev rules...
Failed to send reload request: No such file or directory
Udev rules have been successfully installed.


https://docs.openvino.ai/2023.3/omz_demos_classification_demo_python.html

## **Make inference with NS2**

In [8]:
%cd /tmp/

/tmp


In [9]:
!omz_downloader --name googlenet-v1 --precisions FP16

################|| Downloading googlenet-v1 ||################

========== Downloading /tmp/public/googlenet-v1/googlenet-v1.prototxt
... 100%, 35 KB, 53503 KB/s, 0 seconds passed

========== Downloading /tmp/public/googlenet-v1/googlenet-v1.caffemodel
... 100%, 52279 KB, 13945 KB/s, 3 seconds passed

========== Replacing text in /tmp/public/googlenet-v1/googlenet-v1.prototxt



In [14]:
!pip install numpy==1.19.5

You should consider upgrading via the '/usr/bin/python3.8 -m pip install --upgrade pip' command.


In [10]:
!omz_converter --name googlenet-v1 --precision FP16

========== Converting googlenet-v1 to IR (FP16)
Conversion command: /usr/bin/python3.8 -- /usr/local/bin/mo --framework=caffe --data_type=FP16 --output_dir=/tmp/public/googlenet-v1/FP16 --model_name=googlenet-v1 --input=data '--mean_values=data[104.0,117.0,123.0]' --output=prob --input_model=/tmp/public/googlenet-v1/googlenet-v1.caffemodel --input_proto=/tmp/public/googlenet-v1/googlenet-v1.prototxt '--layout=data(NCHW)' '--input_shape=[1, 3, 224, 224]'

Model Optimizer arguments:
Common parameters:
	- Path to the Input Model: 	/tmp/public/googlenet-v1/googlenet-v1.caffemodel
	- Path for generated IR: 	/tmp/public/googlenet-v1/FP16
	- IR output name: 	googlenet-v1
	- Log level: 	ERROR
	- Batch: 	Not specified, inherited from the model
	- Input layers: 	data
	- Output layers: 	prob
	- Input shapes: 	[1, 3, 224, 224]
	- Source layout: 	Not specified
	- Target layout: 	Not specified
	- Layout: 	data(NCHW)
	- Mean values: 	data[104.0,117.0,123.0]
	- Scale values: 	Not specified
	- Scale fa

In [11]:
!curl -O https://storage.openvinotoolkit.org/data/test_data/images/car_1.bmp

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100 1398k  100 1398k    0     0  2285k      0 --:--:-- --:--:-- --:--:-- 2281k


In [35]:
cd /opt/intel/openvino_2022.1.0.643

/opt/intel/openvino_2022.1.0.643


In [36]:
!python3 samples/python/hello_classification/hello_classification.py /tmp/public/googlenet-v1/FP16/googlenet-v1.xml /tmp/car_1.bmp MYRIAD

[ INFO ] Creating OpenVINO Runtime Core
[ INFO ] Reading the model: /tmp/public/googlenet-v1/FP16/googlenet-v1.xml
[ INFO ] Loading the model to the plugin
[ INFO ] Starting inference in synchronous mode
[ INFO ] Image path: /tmp/car_1.bmp
[ INFO ] Top 10 results: 
[ INFO ] class_id probability
[ INFO ] --------------------
[ INFO ] 656      0.7211914
[ INFO ] 654      0.0698242
[ INFO ] 468      0.0361938
[ INFO ] 817      0.0226746
[ INFO ] 436      0.0221100
[ INFO ] 581      0.0209351
[ INFO ] 705      0.0206146
[ INFO ] 575      0.0159302
[ INFO ] 734      0.0115738
[ INFO ] 511      0.0098953
[ INFO ] 
[ INFO ] This sample is an API example, for any performance measurements please use the dedicated benchmark_app tool



## **Make inference with NS2 with Pytorch Model**

In [24]:
resnet50 = models.resnet50(pretrained=True)

Downloading: "https://download.pytorch.org/models/resnet50-19c8e357.pth" to /root/.cache/torch/hub/checkpoints/resnet50-19c8e357.pth
Widget Javascript not detected.  It may not be installed or enabled properly.


AttributeError: 'FloatProgress' object has no attribute 'style'

**Initialize OpenVINO Runtime Core**

In [3]:
import numpy

In [4]:
numpy.__version__

'1.19.5'

In [5]:
import openvino as ov

In [6]:
from openvino.runtime import Core, Layout, Type
from openvino.preprocess import PrePostProcessor, ResizeAlgorithm
core = Core()

**Initialize OpenVINO Runtime Core**

In [7]:
core.available_devices

['CPU', 'GNA', 'MYRIAD']

In [30]:
# https://docs.openvino.ai/2025/get-started/learn-openvino/openvino-samples/hello-query-device.html
def param_to_string(parameters) -> str:
    """Convert a list / tuple of parameters returned from OV to a string."""
    if isinstance(parameters, (list, tuple)):
        return ', '.join([str(x) for x in parameters])
    else:
        return str(parameters)


def hello_query_device():
    log.basicConfig(format='[ %(levelname)s ] %(message)s', level=log.INFO, stream=sys.stdout)

    # --------------------------- Step 1. Initialize OpenVINO Runtime Core --------------------------------------------
    core = Core()

    # --------------------------- Step 2. Get metrics of available devices --------------------------------------------
    log.info('Available devices:')
    for device in core.available_devices:
        log.info(f'{device} :')
        log.info('\tSUPPORTED_PROPERTIES:')
        for property_key in core.get_property(device, 'SUPPORTED_PROPERTIES'):
            if property_key != 'SUPPORTED_PROPERTIES':  # Skip the meta-property
                try:
                    property_val = core.get_property(device, property_key)
                    log.info(f'\t\t{property_key}: {param_to_string(property_val)}')
                except RuntimeError as e:  # Changed from TypeError to RuntimeError
                    log.info(f'\t\t{property_key}: UNSUPPORTED TYPE ({str(e)})')

    # -----------------------------------------------------------------------------------------------------------------
    return 0

In [31]:
hello_query_device()

[ INFO ] Available devices:
[ INFO ] CPU :
[ INFO ] 	SUPPORTED_PROPERTIES:
[ INFO ] 		AVAILABLE_DEVICES: 
[ INFO ] 		RANGE_FOR_ASYNC_INFER_REQUESTS: 1, 1, 1
[ INFO ] 		RANGE_FOR_STREAMS: 1, 32
[ INFO ] 		FULL_DEVICE_NAME: Intel(R) Core(TM) i9-14900HX
[ INFO ] 		OPTIMIZATION_CAPABILITIES: FP32, FP16, INT8, BIN, EXPORT_IMPORT
[ INFO ] 		CACHE_DIR: 
[ INFO ] 		NUM_STREAMS: 1


TypeError: Unable to convert function return value to a Python type! The signature was
	(self: openvino.pyopenvino.Core, device_name: str, name: str) -> object

In [8]:
%cd /root/workspace/

/root/workspace


In [9]:
net = core.read_model("resnet50_opset11.onnx")

In [10]:
image_path = "/tmp/car_1.bmp"
image = cv2.imread(image_path)
input_tensor = np.expand_dims(image, 0)

In [11]:
ppp = PrePostProcessor(net)

_, h, w, _ = input_tensor.shape

ppp.input().tensor() \
        .set_element_type(Type.u8) \
        .set_layout(Layout('NHWC')) \
        .set_spatial_static_shape(h, w)

ppp.input().preprocess().resize(ResizeAlgorithm.RESIZE_LINEAR)

ppp.input().model().set_layout(Layout('NCHW'))

ppp.output().tensor().set_element_type(Type.f32)

model = ppp.build()

In [12]:
compiled_model = core.compile_model(model, "MYRIAD")

In [47]:
start = time.time()
results = compiled_model.infer_new_request({0: input_tensor})
finish = time.time()
print(f"{(finish - start) * 10e3} ms")

728.7883758544922 ms


In [13]:
%%timeit
results = compiled_model.infer_new_request({0: input_tensor})

72 ms ± 387 µs per loop (mean ± std. dev. of 7 runs, 10 loops each)


In [53]:
!git clone --recurse-submodules https://github.com/openvinotoolkit/open_model_zoo.git

/usr/bin/sh: 1: git: not found


In [54]:
%cd /root/workspace/open_model_zoo/demos/classification_demo/python

/root/workspace/open_model_zoo/demos/classification_demo/python


In [55]:
!python3 classification_demo.py -i 0

Traceback (most recent call last):
  File "classification_demo.py", line 32, in <module>
    from model_api.adapters import create_core, OpenvinoAdapter, OVMSAdapter
  File "/root/workspace/open_model_zoo/demos/common/python/model_zoo/model_api/adapters/__init__.py", line 18, in <module>
    from .openvino_adapter import create_core, OpenvinoAdapter
  File "/root/workspace/open_model_zoo/demos/common/python/model_zoo/model_api/adapters/openvino_adapter.py", line 27, in <module>
    from .utils import Layout
  File "/root/workspace/open_model_zoo/demos/common/python/model_zoo/model_api/adapters/utils.py", line 18, in <module>
    from openvino import layout_helpers
ImportError: cannot import name 'layout_helpers' from 'openvino' (/opt/intel/openvino/python/python3.8/openvino/__init__.py)
